In [ ]:
%pylab inline
import eucare as ec

In [ ]:
import eucare as eu
from eucare.example_tilesets import *
from eucare.example_graphs import *
from eucare.classifiers import *
from eucare import conway
from copy import deepcopy
import ipywidgets as widgets

In [ ]:
plotting_kwargs = {
#     'figsize': (10, 10),
    'render_faces': True,
    'render_vertices': False,
    'render_edges': False,
    'face_inset': 0.02,
#     'line_width': 3,
}
def show(G):
    cc = congruency_classifier()

    for f in G.faces:
        f['color_key'] = cc.classify(f)
    
    G.show(**plotting_kwargs)

In [ ]:
for n in (3,4,6):
    G = from_tiles(eu.example_tilesets.platonic(n), rings=3, vertex_based=True)
    show(G)

In [ ]:
G = from_tiles(eu.example_tilesets.t_4_6_12(), rings=3)
# G = from_tiles(eu.example_tilesets.platonic(6), rings=3)
    
show(G)

#G = conway.gyro_graph()(G)
#show(G)

# G = conway.chamfer_graph()(G)
# show()

operators = [
    conway.dual_graph, 
    conway.truncate_graph, 
    conway.ambo_graph, 
    conway.join_graph, 
    conway.kis_graph, 
    conway.gyro_graph,
    conway.loft_graph,
    conway.chamfer_graph,
    conway.starify_graph,
    conway.goldberg2_graph
]

def operator_name(operator):
    return operator.__name__.split('_')[0]

operator_options = [(operator_name(op), op) for op in operators]
    
def f(operator):
    print(operator_name(operator))
    show(operator()(G.copy()))
    
widgets.interact(f,
                operator=widgets.RadioButtons(options=operator_options));
#G = conway.ambo_graph()(conway.gyro_graph()(G))
#show()

In [ ]:
def f2(operator1, operator2):
    #print(f'{operator_name(operator1)} then {operator_name(operator2)}')
    show(operator2()(operator1()(G.copy())))
    
widgets.interact(f2,
                operator1=widgets.Dropdown(description='First', options=operator_options),
                operator2=widgets.Dropdown(description='then', options=operator_options))

In [ ]:
def central_face(G):
    fs = list(G.faces)
    return fs[np.argmin([np.linalg.norm(f.midpoint()) for f in fs])]

def central_vertex(G):
    vs = list(G.vertices)
    return vs[np.argmin([np.linalg.norm(v['pos']) for v in vs])]

# G = from_tiles(eu.example_tilesets.t_4_6_12(), rings=3)
G = from_tiles(eu.example_tilesets.platonic(6), rings=4)

G = conway.kis_graph()(G, faces=[central_face(G)])
v = central_vertex(G)
[G.delete_edge(e.nex) for e in v.outgoing_iter()]

show(G)
ps, vs = G.get_position_view()
#ps[:, 1] *= 1.5
def rot_mat(alpha):
    return np.array([[np.cos(alpha), np.sin(alpha)],[-np.sin(alpha), np.cos(alpha)]])
#ps[:] = ps @ rot_mat(np.pi/12)
k = ps.copy()
k = np.array([complex(*ki) for ki in k])
k -= np.mean(k)
k = k**2
k = np.stack([k.real, k.imag], axis=-1)
k /= np.std(k)
ps[:] = k
#ps[:] = [p @ rot_mat(np.linalg.norm(p) / 10) for p in ps]

G.recompute_lengths_and_angles()

# G = conway.dual_graph()(G)
show(G)